# 17 — E-round analysis: the v0.13 verdicts (zero solves; runs after 16)

E12 bracket, E17-T3 counterfactual leans vs pre-stated expectations, E8/E9/E10 verdicts,
E15-if-run. Every block prints results_log-R9-ready numbers. Kernel `y2y-geo`.

In [6]:
import importlib, json, pathlib, sys

import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from pyproj import Transformer

_cands = [p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config.py").exists()]
ROOT = _cands[0]
sys.path.insert(0, str(ROOT))
import config, leverage_core as lc, ensemble_core as ec
for _m in (config, lc, ec):
    importlib.reload(_m)
SPEC = ROOT / "analyses" / "y2y" / "spec"
RUNS = ROOT / "analyses" / "y2y" / "runs"
FIG = ROOT / "analyses" / "y2y" / "figures"
ER = json.loads((SPEC / "e_round_v13.json").read_text())
MAN = pd.read_csv(SPEC / "manifest.csv").set_index("formulation_id")
pu = lc.pu_mask()
with rasterio.open(config.HANDOFF_DIR / "mask_protected_areas.tif") as src:
    locked2d = (src.read(1) == 1) & pu
disc = ~locked2d[pu]
msoc = lc._read(config.HANDOFF_DIR / "irrecoverable_carbon_m_soc.tif")
vp_msoc = np.nan_to_num(msoc[pu], nan=0.0)
def rep_of(rel):
    return pd.read_csv(ROOT / rel / "run" / "portfolio_representation.csv"
                       ).set_index("feature")["relative_held"]
def sel_of(rel):
    return ec.read_selections(ROOT / rel / "run" / "portfolio.tif", pu)[0]
print("E15 was", "TRIGGERED" if ER["e14"]["trigger"] else "not triggered")

E15 was TRIGGERED


In [7]:
# ---- E12: the estimator bracket ------------------------------------------------------------
for fid in ER["e12"]["formulations"]:
    A = ec.read_selections(RUNS / fid / "anchor.tif", pu)[0]
    mga = np.vstack([A[None, :], ec.read_selections(RUNS / fid / "mga_g05.tif", pu)])
    maa = np.vstack([A[None, :], ec.read_selections(RUNS / fid / "maa_g05.tif", pu)])
    f1, f2 = mga.mean(axis=0), maa.mean(axis=0)
    m_disc = int(A[disc].sum())
    stats = {}
    for nm, S in (("MGA", mga), ("MAA", maa)):
        Sd = S[:, disc].astype(np.float32)
        sizes = Sd.sum(axis=1)
        ham = sizes[:, None] + sizes[None, :] - 2 * (Sd @ Sd.T)
        f = S.mean(axis=0)[disc]
        stats[nm] = dict(D=float(ham.max() / (2 * m_disc)),
                         C=float((f == 1).sum() / m_disc),
                         freq_km2=int((f >= 0.7).sum()),
                         within_var=float(np.mean(f * (1 - f))))
    corr = float(np.corrcoef(f1[disc], f2[disc])[0, 1])
    print(f"{fid}: corr(f_MGA, f_MAA) = {corr:.3f}")
    for nm, st in stats.items():
        print(f"   {nm}: D {st['D']:.3f} | C {st['C']:.3f} | frequent {st['freq_km2']:,} km2 "
              f"| within-var {st['within_var']:.4f}")
print("\nbracket reading: if the two estimators agree on f (corr high) and bands, the "
      "estimator-conditionality caveat is narrow -> full-14 MAA unnecessary; report bracket.")

s0_ssp585_theta5: corr(f_MGA, f_MAA) = 0.916
   MGA: D 0.953 | C 0.020 | frequent 11,247 km2 | within-var 0.1316
   MAA: D 0.753 | C 0.019 | frequent 10,796 km2 | within-var 0.1290
s2_ssp585_theta5: corr(f_MGA, f_MAA) = 0.851
   MGA: D 0.979 | C 0.011 | frequent 4,748 km2 | within-var 0.1381
   MAA: D 0.789 | C 0.011 | frequent 4,882 km2 | within-var 0.1354
s4_ssp585_theta3: corr(f_MGA, f_MAA) = 0.964
   MGA: D 0.875 | C 0.073 | frequent 28,748 km2 | within-var 0.1155
   MAA: D 0.660 | C 0.073 | frequent 30,130 km2 | within-var 0.1132

bracket reading: if the two estimators agree on f (corr high) and bands, the estimator-conditionality caveat is narrow -> full-14 MAA unnecessary; report bracket.


In [8]:
# ---- E17-T3: leave-one-block-out latitudinal shifts vs pre-stated expectations -------------
with rasterio.open(config.HANDOFF_DIR / "cost_uniform.tif") as src:
    tr = src.transform
rows_, cols_ = np.where(pu)
xs = tr.c + (cols_ + 0.5) * tr.a
ys = tr.f + (rows_ + 0.5) * tr.e
_, lat = Transformer.from_crs(config.TARGET_CRS, "EPSG:4326", always_xy=True).transform(xs, ys)
LAT = lat.astype(np.float32)
s0 = ec.read_selections(RUNS / "s0_ssp585_theta5" / "anchor.tif", pu)[0]
base_lat = float(LAT[s0 & disc].mean())
print(f"S0 anchor mean discretionary latitude: {base_lat:.2f}N\n")
outs = list(ER["e17_t3"]["blocks"]) + ["efg"]
for b in outs:
    d = f"analyses/y2y/runs/e17_t3/{b}_out"
    try:
        sel = sel_of(d)
    except Exception:
        print(f"{b}_out: missing -- run 16"); continue
    ml = float(LAT[sel & disc].mean())
    jac = float((sel & s0).sum() / (sel | s0).sum())
    exp = ER["e17_t3"]["expectations"].get(b, "")
    print(f"{b + '_out':<20} mean lat {ml:.2f}N ({ml - base_lat:+.2f}) | Jaccard vs S0 {jac:.3f}"
          f"\n{'':<20} expected: {exp}")

S0 anchor mean discretionary latitude: 50.96N

core_habitat_out     mean lat 50.79N (-0.17) | Jaccard vs S0 0.680
                     expected: shifts toward richness/carbon country (refugia mass is northern-interior)
connectivity_out     mean lat 50.01N (-0.96) | Jaccard vs S0 0.833
                     expected: small shift (diffuse)
carbon_out           mean lat 49.78N (-1.18) | Jaccard vs S0 0.885
                     expected: selection shifts SOUTH-of-current? pre-stated: south (tail is southern-interior)
biodiversity_out     mean lat 52.14N (+1.18) | Jaccard vs S0 0.818
                     expected: selection shifts NORTH (AOH mass is southern)
efg_out              mean lat 53.07N (+2.11) | Jaccard vs S0 0.695
                     expected: UNKNOWN -- that is the point of the run


In [9]:
# ---- E8 / E9 / E10 verdicts ----------------------------------------------------------------
print("== E8: m_soc x10 inertness (expect ~unchanged)")
for fid, tag in (("s0_ssp585_theta5", "s0"), ("s3_ssp585_theta5", "s3")):
    a = ec.read_selections(RUNS / fid / "anchor.tif", pu)[0]
    try:
        x = sel_of(f"analyses/y2y/runs/e8/{tag}_carbonx10")
    except Exception:
        print(f"  {tag}: missing"); continue
    print(f"  {tag}: Jaccard vs anchor {float((a & x).sum() / (a | x).sum()):.4f} | "
          f"m_soc capture {float((x @ (vp_msoc / vp_msoc.sum()))):.4f} (anchor "
          f"{float((a @ (vp_msoc / vp_msoc.sum()))):.4f})")

print("\n== E9: target lever vs weights-only vs log-carbon (per-hectare prediction)")
dec9 = vp_msoc >= np.quantile(vp_msoc[vp_msoc > 0], 0.9)      # densest decile of positive m_soc
for arm in ("infweights", "logcarbon"):
    try:
        x = sel_of(f"analyses/y2y/runs/e9/{arm}")
    except Exception:
        print(f"  {arm}: missing"); continue
    cap = float(x @ (vp_msoc / vp_msoc.sum()))
    dcap = float(vp_msoc[dec9 & x].sum() / vp_msoc[dec9].sum())
    print(f"  {arm:<12} m_soc capture {cap:.3f} | densest-decile mass capture {dcap:.3f}")
s0cap = float(s0 @ (vp_msoc / vp_msoc.sum()))
d0 = float(vp_msoc[dec9 & s0].sum() / vp_msoc[dec9].sum())
print(f"  {'S0 (target)':<12} m_soc capture {s0cap:.3f} | densest-decile mass capture {d0:.3f}")

print("\n== E10: capture vs theta (targets should park at the kink)")
for arm, t in (("theta10", 0.121), ("theta3", 0.552)):
    try:
        r = rep_of(f"analyses/y2y/runs/e10/{arm}")
    except Exception:
        print(f"  {arm}: missing"); continue
    print(f"  {arm}: m_soc capture {float(r['irrecoverable_carbon_m_soc']):.4f} (target {t})")
print(f"  theta5 (S0): 0.3320 (target 0.332) -- the frozen reference")

== E8: m_soc x10 inertness (expect ~unchanged)
  s0: Jaccard vs anchor 0.9962 | m_soc capture 0.3320 (anchor 0.3320)
  s3: Jaccard vs anchor 0.9935 | m_soc capture 0.3320 (anchor 0.3320)

== E9: target lever vs weights-only vs log-carbon (per-hectare prediction)
  infweights   m_soc capture 0.462 | densest-decile mass capture 0.632
  logcarbon    m_soc capture 0.266 | densest-decile mass capture 0.236
  S0 (target)  m_soc capture 0.332 | densest-decile mass capture 0.392

== E10: capture vs theta (targets should park at the kink)
  theta10: m_soc capture 0.2290 (target 0.121)
  theta3: m_soc capture 0.5520 (target 0.552)
  theta5 (S0): 0.3320 (target 0.332) -- the frozen reference


In [10]:
# ---- E15 (if run): guardrailed vs plain band ----------------------------------------------
if not ER["e14"]["trigger"]:
    print("E15 not triggered (E14 verdict) -- nothing to analyze; the aggregate band is "
          "already a de-facto per-block band at g=5%.")
else:
    for fid in ("s0_ssp585_theta5", "s4_ssp585_theta3"):
        p_ = RUNS / fid / "mga_guard_g05.tif"
        if not p_.exists():
            print(f"{fid}: guard members missing -- run 16"); continue
        A = ec.read_selections(RUNS / fid / "anchor.tif", pu)[0]
        for nm, tif in (("plain", "mga_g05.tif"), ("block-guarded", "mga_guard_g05.tif"),
                ("value-guarded", "mga_guardfeat_g05.tif")):
            if not (RUNS / fid / tif).exists():
                print(f"  {fid} {nm}: missing"); continue
            S = np.vstack([A[None, :], ec.read_selections(RUNS / fid / tif, pu)])
            Sd = S[:, disc].astype(np.float32)
            sizes = Sd.sum(axis=1)
            ham = sizes[:, None] + sizes[None, :] - 2 * (Sd @ Sd.T)
            m_disc = int(A[disc].sum())
            f = S.mean(axis=0)[disc]
            print(f"  {fid} {nm:<14} D {float(ham.max()/(2*m_disc)):.3f} | "
                  f"C {float((f==1).sum()/m_disc):.3f} | frequent {int((f>=0.7).sum()):,} km2")

  s0_ssp585_theta5 plain          D 0.953 | C 0.020 | frequent 11,247 km2
  s0_ssp585_theta5 block-guarded  D 0.913 | C 0.042 | frequent 23,108 km2
  s0_ssp585_theta5 value-guarded  D 0.909 | C 0.044 | frequent 23,996 km2
  s4_ssp585_theta3 plain          D 0.875 | C 0.073 | frequent 28,748 km2
  s4_ssp585_theta3 block-guarded  D 0.854 | C 0.073 | frequent 34,787 km2
  s4_ssp585_theta3 value-guarded  D 0.850 | C 0.072 | frequent 34,427 km2
